In [1]:
%load_ext autoreload
%autoreload 2

from src import loaders

In [3]:
mc_sl_2x2 = "tests/snake_ladder/mc_2x2.pm"
mon_sl_2x2 = "tests/snake_ladder/mon_2x2.pm"

mc_sl_2x2_deadlock = "tests/snake_ladder/mc_2x2_deadlock.pm"
mon_sl_2x2_deadlock = "tests/snake_ladder/mon_2x2_deadlock.pm"

mc_sl_nxn = "tests/snake_ladder/mc_nxn.pm"
mon_sl_nxn = "tests/snake_ladder/mon_nxn.pm"

mc_sl_u_nxn = "tests/snake_ladder/mc_u_nxn.pm"

mc_simple = "tests/simple/mc1.pm"
mon_simple = "tests/simple/mon1.pm"

gb_r = "tests/gb_reachability.pm"
gb_e = "tests/gb_exact.pm"

# mc, n, ladders, snakes = loaders.load_defined_snl(mc_sl_nxn)
mon = loaders.load_dfa(mon_sl_nxn)
gb = loaders.load_dfa(gb_e)

In [15]:
from random import randrange
from math import sqrt

def random_ladder(n):
    source = randrange(1,n-int(sqrt(n)))
    dest = randrange(source+int(sqrt(n)), int(min(n, source + n / 2)))
    return source, dest

def random_snake(n):
    source = randrange(int(sqrt(n))+1,n)
    dest = randrange(1, source-int(sqrt(n)))
    return source, dest

n = 5**2
ladders = dict(random_ladder(n) for _ in range(int(sqrt(n))))
snakes = dict(random_snake(n) for _ in range(int(sqrt(n))))

# Random snakes and ladders
mc = loaders.load_snl(mc_sl_u_nxn, n, ladders, snakes)

milton_snakes = {98: 76, 95: 75, 93: 73, 87: 24, 64: 60, 62: 19, 55: 53, 49: 11, 47: 26, 16: 6}
milton_ladders = {1: 38, 4: 14, 9: 31, 28: 64, 40: 42, 36: 44, 51: 67, 71: 91, 80: 100}
# mc = loaders.load_snl(mc_sl_u_nxn, n := 10**2, ladders:=milton_ladders, snakes:=milton_snakes)



In [16]:
from stormvogel.show import show
# show(gb)
show(mon)
show(mc)
# print(mc)

Output()

Output()

In [17]:
from src.MDP_product import MC_MON_Product
from time import time

t = time()
pomdp = MC_MON_Product(mc, mon, gb, "good")
pomdp.apply_spec('P>=0.5 [ F<=3 "good"]')
pomdp.create_product()
# pomdp.add_ret_to_bmecs()
# pomdp.remove_unreachable_states()
print(f"--------------------\nStarting Paynt after {time() - t}s")
assignment = pomdp.check_paynt_prop('Pmax=? [ (F "goal") ]')

New good states become: [['[pos=25]']]
--------------------
Starting Paynt after 0.040682077407836914s
> progress 36.572%, elapsed 3 s, estimated 8 s, iters = {MDP: 2525}, opt = 0.9523
> progress 83.984%, elapsed 6 s, estimated 7 s, iters = {MDP: 5197}, opt = 0.9523
--------------------
Synthesis summary:
optimality objective: Pmax=? [F "goal"] 

method: AR, synthesis time: 6.3 s
number of holes: 21, family size: 1e12, quotient: 1094 states / 4318 actions
explored: 100 %
MDP stats: avg MDP size: 210, iterations: 5495

optimum: 0.952344
--------------------
counterexample found:  A(0,0)=normal, A(1,0)=ladder, A(2,0)=normal, A(3,0)=normal, A(4,0)=normal, A(5,0)=normal, A(6,0)=normal, A(7,0)=normal, A(8,0)=normal, A(9,0)=normal, A(10,0)=normal, A(11,0)=normal, A(12,0)=normal, A(13,0)=normal, A(14,0)=normal, A(15,0)=normal, A(16,0)=normal, A(17,0)=normal, A(18,0)=normal, A(19,0)=normal, A(20,0)=end 
--------------------


In [22]:
%matplotlib notebook
from src.draw import animate_player_movement
import math
from IPython.display import HTML

induced_mc, poss = pomdp.simulate_paynt_assignment(assignment, 100)

goal_squares = [next(int(l[5:-1]) for l in state.labels if l.startswith("[pos")) 
                for state in pomdp.mc.states.values() 
                if "good" in state.labels]
player_path = poss

animation = animate_player_movement(int(math.sqrt(n)), snakes, ladders, goal_squares, player_path)
HTML(animation.to_jshtml())

s0, labels=!l0 !s0 init [pos=0] !g0

	--[0, 0:normal]-->
s58, labels=!g0 !l1 !s4 [pos=4] accepting normal
	--[1, 1:ladder]-->
s114, labels=!g0 !l2 !s8 [pos=8] accepting ladder
	--[2, 0:normal]-->
s172, labels=!g0 !l3 !s14 [pos=19] accepting normal
	--[3, 0:normal]-->
s230, labels=!g0 !l4 !s20 [pos=20] accepting normal
	--[4, 0:normal]-->
s285, labels=!g0 !l5 !s23 [pos=23] accepting normal
	--[5, 0:normal]-->
s337, labels=!g0 !l6 !s23 [pos=23] accepting normal
	--[6, 0:normal]-->
s388, labels=!g0 !l7 !s22 [pos=22] accepting normal
	--[7, 0:normal]-->
s469, labels=!g1 !l8 !s25 [pos=25] accepting good normal
	--[8, 0:normal]-->
s521, labels=!g1 !l9 !s25 [pos=25] accepting good normal
	--[9, 0:normal]-->
s573, labels=!g1 !l10 !s25 [pos=25] accepting good normal
	--[10, 0:normal]-->
s625, labels=!g1 !l11 !s25 [pos=25] accepting good normal
	--[11, 0:normal]-->
s677, labels=!g1 !l12 !s25 [pos=25] accepting good normal
	--[12, 0:normal]-->
s729, labels=!g1 !l13 !s25 [pos=25] accepting good no

<IPython.core.display.Javascript object>

In [26]:
from stormvogel.show import show
from stormvogel.mapping import stormpy_to_stormvogel
from stormpy import model_checking, parse_properties

imc =  stormpy_to_stormvogel(induced_mc)
result_goal = model_checking(induced_mc, parse_properties('Pmax=? [F "goal"]')[0])
result_stop = model_checking(induced_mc, parse_properties('Pmax=? [F "stop"]')[0])
prop_goal = result_goal.at(induced_mc.initial_states[0])
prop_stop = result_stop.at(induced_mc.initial_states[0])
print(f"probability to reach goal={prop_goal} and stop={prop_stop}. Total={prop_goal+prop_stop}")
show(imc)


probability to reach goal=0.9533725060927631 and stop=0.046627493907236. Total=0.9999999999999991


Output()

In [ ]:
# scheduler = pomdp.check_storm_prop('Pmax=? [ F "goal" ]')